# Interactive User-Preference GA for Chair Design

**Workflow**
1. Load pretrained PartSDF (`model_2000.pth`).
2. At each generation, generate **8 candidate chairs**.
3. Use **interactive sliders/buttons** to rate each chair 1-5.
4. Click **"Evolve → Next Generation"** to run selection/crossover/mutation.
5. Repeat for as many generations as you like.
6. Export the best chairs.

This is an **interactive** GA — you are the fitness function.

## 0. Setup

In [1]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

def _find_partsdf_root(start_dir):
    """Walk up from start_dir until we find a folder containing experiments/chair/specs.json."""
    cur = os.path.abspath(start_dir)
    for _ in range(10):  # max 10 levels up
        if os.path.isfile(os.path.join(cur, "experiments", "chair", "specs.json")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    return None

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PARTSDF_ROOT = "/content/drive/MyDrive/PartSDF"
    !pip install -q trimesh scikit-image matplotlib numpy torch libigl ipywidgets
else:
    # Locate PartSDF root robustly (works regardless of current cwd / notebook state)
    PARTSDF_ROOT = _find_partsdf_root(os.getcwd())
    if PARTSDF_ROOT is None:
        # Fallback: hard-coded (edit if needed)
        PARTSDF_ROOT = "/home/user/bodyawarechair/PartSDF"

assert os.path.isdir(PARTSDF_ROOT), f"PARTSDF_ROOT not found: {PARTSDF_ROOT}"
assert os.path.isfile(os.path.join(PARTSDF_ROOT, "experiments", "chair", "specs.json")), \
    f"specs.json not found under {PARTSDF_ROOT}/experiments/chair/"

if PARTSDF_ROOT not in sys.path:
    sys.path.insert(0, PARTSDF_ROOT)
os.chdir(PARTSDF_ROOT)

print(f"PARTSDF_ROOT: {PARTSDF_ROOT}")
print(f"Working directory: {os.getcwd()}")

PARTSDF_ROOT: /home/user/bodyawarechair/PartSDF
Working directory: /home/user/bodyawarechair/PartSDF


In [2]:
%matplotlib inline

import json
import copy
import random
import io
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import trimesh
import torch

import ipywidgets as widgets
from IPython.display import display, clear_output

from src import workspace as ws
from src.model import get_model, get_part_latents, get_part_poses
from src.mesh import create_mesh, create_parts, SdfGridFiller
from src.utils import get_device, set_seed, get_color_parts
from src.primitives import standardize_quaternion

device = get_device()
print(f"Device: {device}")

Device: cuda:0


/home/user/bodyawarechair/PartSDF/src/visualization.py:33: UserWarning: PyTorch3D not found, rendering functions will not work.
  warnings.warn("PyTorch3D not found, rendering functions will not work.")


## 1. Load Pretrained PartSDF (`model_2000.pth`)

In [3]:
expdir = os.path.join(PARTSDF_ROOT, "experiments", "chair")
specs = ws.load_specs(expdir)

n_parts  = specs["Parts"]["NumParts"]
part_dim = specs["Parts"]["LatentDim"]
clampD   = specs["ClampingDistance"]
use_poses = specs["Parts"].get("UsePoses", False)
use_occ   = specs.get("ImplicitField", "SDF").lower() in ["occ", "occupancy"]

latent_ckpt = torch.load(os.path.join(expdir, "latent", "latents_2000.pth"), map_location="cpu")
n_shapes = latent_ckpt["weight"].shape[0]

instances = [f"chair_{i:04d}" for i in range(n_shapes)]
train_split_path = specs.get("TrainSplit", None)
if train_split_path:
    # TrainSplit is relative to PARTSDF_ROOT in specs.json
    if not os.path.isabs(train_split_path):
        train_split_path = os.path.join(PARTSDF_ROOT, train_split_path)
    if os.path.isfile(train_split_path):
        with open(train_split_path) as f:
            instances = json.load(f)

print(f"{n_shapes} training shapes, {n_parts} parts, part_dim={part_dim}, use_poses={use_poses}")

model = get_model(
    specs["Network"], **specs.get("NetworkSpecs", {}),
    n_parts=n_parts, part_dim=part_dim, use_occ=use_occ,
).to(device)
latents = get_part_latents(n_shapes, n_parts, part_dim, specs.get("LatentBound", None), device=device)
if use_poses:
    poses = get_part_poses(n_shapes, n_parts, freeze=True, device=device, fill_nans=True)

logs = ws.load_history(expdir)
cp_epoch = logs["epoch"]  # 2000
ws.load_model(expdir, model, cp_epoch)
ws.load_latents(expdir, latents, cp_epoch)
if use_poses:
    ws.load_poses(expdir, poses)

model.eval()
for p in model.parameters():
    p.requires_grad_(False)
latents.requires_grad_(False)

print(f"Loaded checkpoint epoch={cp_epoch}")
grid_filler = SdfGridFiller(128, device)

/tmp/ipykernel_2553935/3655030125.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  latent_ckpt = torch.load(os.path.join(expdir, "latent", "latents_2000.pth"), map_locat

1065 training shapes, 8 parts, part_dim=256, use_poses=True
Loaded checkpoint epoch=2000


/home/user/bodyawarechair/PartSDF/src/workspace.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  history = torch.load(filename)
/home/user/bodyawarechair/PartSDF/src/wo

## 2. Part Library

In [4]:
part_latents_all = latents.weight.detach().clone()  # (n_shapes, n_parts, latent_dim)

if use_poses:
    part_poses_all = poses.weight.detach().clone()  # (n_shapes, n_parts, 10)

    def decompose_poses(pose_tensor):
        R = standardize_quaternion(pose_tensor[..., :4])
        t = pose_tensor[..., 4:7]
        s = pose_tensor[..., 7:10]
        return R, t, s

    mean_pose = part_poses_all.nanmean(dim=0, keepdim=True)
    nan_mask = torch.isnan(part_poses_all).any(-1)
    if nan_mask.any():
        for p in range(n_parts):
            idx = nan_mask[:, p]
            part_poses_all[idx, p] = mean_pose[0, p]

print(f"Part library: latents={part_latents_all.shape}")
if use_poses:
    print(f"              poses={part_poses_all.shape}")

Part library: latents=torch.Size([1065, 8, 256])
              poses=torch.Size([1065, 8, 10])


## 3. Chromosome & Decoding

Using **intrinsic mode**: parts' latents come from different chairs, but **poses are anchored to one chair** to keep parts in correct spatial positions (prevents floating parts).

In [5]:
MIXING_MODE = "intrinsic"  # "intrinsic" or "full"

@dataclass
class Chromosome:
    genes: List[int]                 # length n_parts; which chair for each part's LATENT
    pose_anchor: int = 0             # chair index used for ALL pose values (intrinsic mode)
    fitness: float = 0.0             # user rating 1-5 (0 = not rated yet)
    fitness_details: dict = field(default_factory=dict)

    def __repr__(self):
        return f"Chr(genes={self.genes}, anchor={self.pose_anchor}, rating={self.fitness})"


def random_chromosome():
    return Chromosome(
        genes=[random.randint(0, n_shapes - 1) for _ in range(n_parts)],
        pose_anchor=random.randint(0, n_shapes - 1),
    )


def chromosome_to_latent_pose(chrom: Chromosome):
    lat = torch.stack([
        part_latents_all[chrom.genes[p], p] for p in range(n_parts)
    ], dim=0).unsqueeze(0).to(device)

    if use_poses:
        if MIXING_MODE == "intrinsic":
            pose = part_poses_all[chrom.pose_anchor].unsqueeze(0).to(device)
        else:
            pose = torch.stack([
                part_poses_all[chrom.genes[p], p] for p in range(n_parts)
            ], dim=0).unsqueeze(0).to(device)
        R, t, s = decompose_poses(pose)
    else:
        R, t, s = None, None, None

    return lat, R, t, s


def decode_chair(chrom: Chromosome, resolution: int = 96):
    lat, R, t, s = chromosome_to_latent_pose(chrom)
    mesh = create_mesh(model, lat, resolution, 32**3,
                       grid_filler=grid_filler, device=device, R=R, t=t, s=s)
    parts = create_parts(model, lat, resolution, 32**3,
                         grid_filler=grid_filler, device=device, R=R, t=t, s=s)
    return mesh, parts


# Test
test_chrom = random_chromosome()
print(f"Test chromosome: {test_chrom}")
m, p = decode_chair(test_chrom, resolution=64)
print(f"Mesh: {len(m.vertices)} verts, {sum(1 for x in p if not x.is_empty)}/{n_parts} active parts")

Test chromosome: Chr(genes=[574, 699, 130, 136, 12, 968, 607, 297], anchor=947, rating=0.0)
Mesh: 26276 verts, 7/8 active parts


## 4. Visualization — render 8 candidates in a grid

In [ ]:
def render_mesh_views(mesh, ax, color_by='height'):
    """Quick matplotlib scatter of mesh vertices — used for preview grids."""
    if mesh.is_empty or len(mesh.vertices) == 0:
        ax.text(0.5, 0.5, "(empty)", ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.set_xticks([]); ax.set_yticks([])
        return
    v = mesh.vertices
    if color_by == 'height':
        c = v[:, 1]
    else:
        c = None
    ax.scatter(v[:, 0], v[:, 1], c=c, cmap='viridis', s=0.3, alpha=0.6)
    ax.set_aspect('equal')
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_xticks([]); ax.set_yticks([])


def render_population(population, resolution=64, title="Generation"):
    """Render all chromosomes in the population in a grid."""
    n = len(population)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
    if rows == 1:
        axes = np.array([axes])

    meshes_cache = []
    for i, chrom in enumerate(population):
        r, c = i // cols, i % cols
        ax = axes[r, c]
        mesh, _ = decode_chair(chrom, resolution=resolution)
        meshes_cache.append(mesh)
        render_mesh_views(mesh, ax)
        rating_str = f"★{chrom.fitness:.1f}" if chrom.fitness > 0 else "not rated"
        ax.set_title(f"#{i}  {rating_str}\nanchor={chrom.pose_anchor}", fontsize=10)

    # Hide extra axes
    for i in range(n, rows * cols):
        r, c = i // cols, i % cols
        axes[r, c].axis('off')

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()
    return meshes_cache

## 5. GA Operators

In [ ]:
def tournament_selection(population, k=3):
    """Pick the highest-rated individual from a random tournament."""
    # Only select from rated individuals
    rated = [c for c in population if c.fitness > 0]
    if not rated:
        return random.choice(population)
    candidates = random.sample(rated, min(k, len(rated)))
    return max(candidates, key=lambda c: c.fitness)


def roulette_selection(population):
    """Fitness-proportionate selection."""
    rated = [c for c in population if c.fitness > 0]
    if not rated:
        return random.choice(population)
    weights = [c.fitness for c in rated]
    return random.choices(rated, weights=weights, k=1)[0]


def crossover_uniform(p1: Chromosome, p2: Chromosome):
    g1, g2 = [], []
    for i in range(n_parts):
        if random.random() < 0.5:
            g1.append(p1.genes[i]); g2.append(p2.genes[i])
        else:
            g1.append(p2.genes[i]); g2.append(p1.genes[i])
    a1 = p1.pose_anchor if random.random() < 0.5 else p2.pose_anchor
    a2 = p2.pose_anchor if random.random() < 0.5 else p1.pose_anchor
    return (Chromosome(genes=g1, pose_anchor=a1),
            Chromosome(genes=g2, pose_anchor=a2))


def crossover_single_point(p1: Chromosome, p2: Chromosome):
    pt = random.randint(1, n_parts - 1)
    return (Chromosome(genes=p1.genes[:pt] + p2.genes[pt:], pose_anchor=p1.pose_anchor),
            Chromosome(genes=p2.genes[:pt] + p1.genes[pt:], pose_anchor=p2.pose_anchor))


def mutate(chrom: Chromosome, mutation_rate: float = 0.2):
    new_genes = chrom.genes.copy()
    new_anchor = chrom.pose_anchor
    for i in range(n_parts):
        if random.random() < mutation_rate:
            new_genes[i] = random.randint(0, n_shapes - 1)
    if random.random() < mutation_rate:
        new_anchor = random.randint(0, n_shapes - 1)
    return Chromosome(genes=new_genes, pose_anchor=new_anchor)


def mutate_swap(chrom: Chromosome):
    g = chrom.genes.copy()
    i, j = random.sample(range(n_parts), 2)
    g[i], g[j] = g[j], g[i]
    return Chromosome(genes=g, pose_anchor=chrom.pose_anchor)


print("GA operators defined.")

## 6. GA Config & Initial Population

In [ ]:
GA_CONFIG = {
    'population_size': 8,         # show 8 chairs per generation
    'elitism_count': 2,           # keep top 2 rated chairs each generation
    'crossover_rate': 0.8,
    'mutation_rate': 0.20,
    'tournament_size': 3,
    'preview_resolution': 64,     # mesh resolution during rating (low = fast)
    'final_resolution': 128,      # mesh resolution for final export
    'seed': 42,
}

set_seed(GA_CONFIG['seed'])

# Generation counter & history
generation = 0
all_rated_history = []    # [(gen, chromosome), ...] — every rated chromosome

# Initial population: 8 random chairs
population = [random_chromosome() for _ in range(GA_CONFIG['population_size'])]

print(f"Generation {generation} initialized: {len(population)} random chairs.")
print(f"Config: {GA_CONFIG}")

## 7. 🎨 Interactive Rating UI

Run this cell **once** to open the interactive rating panel. You'll see:
- 8 chair previews in a grid
- A **slider (1-5)** under each chair for rating
- **"Apply Ratings & Evolve →"** button to move to the next generation
- **"Regenerate Preview"** button to re-render current population (useful after changing resolution)
- **"New Random Population"** to reset

In [ ]:
# ============================================================
# Interactive rating UI (ipywidgets) -- side-view 3D mesh render
# ============================================================
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


def _mesh_to_png(mesh, figsize=(3.2, 3.2), title="", elev=0, azim=-60):
    """Render a trimesh Mesh as a shaded 3D side view (camera tilted down)."""
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    if mesh.is_empty or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        ax.text2D(0.5, 0.5, "(empty)", ha='center', va='center',
                  transform=ax.transAxes, color='red')
    else:
        verts = np.asarray(mesh.vertices).copy()
        # Swap Y <-> Z so matplotlib renders Y as vertical
        verts = verts[:, [0, 2, 1]]

        faces = np.asarray(mesh.faces)
        tris  = verts[faces]

        v0 = tris[:, 1] - tris[:, 0]
        v1 = tris[:, 2] - tris[:, 0]
        n  = np.cross(v0, v1)
        n_norm = np.linalg.norm(n, axis=1, keepdims=True) + 1e-9
        n = n / n_norm

        light_dir = np.array([0.4, 0.8, 0.5])
        light_dir = light_dir / np.linalg.norm(light_dir)
        shade = np.clip((n @ light_dir) * 0.5 + 0.55, 0.25, 1.0)
        colors = np.stack([shade * 0.75, shade * 0.78, shade * 0.82,
                           np.ones_like(shade)], axis=-1)

        pc = Poly3DCollection(tris, facecolors=colors, edgecolors='none',
                              linewidths=0, antialiased=True)
        ax.add_collection3d(pc)

        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
        try:
            ax.set_box_aspect((1, 1, 1))
        except Exception:
            pass

    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=9, pad=0)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight',
                dpi=85, facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def _build_chair_card(idx, chrom, resolution):
    mesh, _ = decode_chair(chrom, resolution=resolution)
    img_bytes = _mesh_to_png(mesh, title=f"#{idx}  anchor={chrom.pose_anchor}")
    img_w = widgets.Image(value=img_bytes, format='png', width=240, height=240)

    initial_val = int(chrom.fitness) if chrom.fitness > 0 else 3
    slider = widgets.IntSlider(
        value=initial_val, min=0, max=5, step=1,
        description=f"#{idx}:", continuous_update=False,
        layout=widgets.Layout(width='240px'),
        style={'description_width': '35px'},
    )
    return widgets.VBox([img_w, slider]), slider


_status_out = widgets.Output()
_grid_out   = widgets.Output()
_plot_out   = widgets.Output()
_winner_out = widgets.Output()
_sliders: List[widgets.IntSlider] = []


def _refresh_grid():
    global _sliders
    _sliders = []
    cards = []
    for i, chrom in enumerate(population):
        card, slider = _build_chair_card(i, chrom, GA_CONFIG['preview_resolution'])
        cards.append(card)
        _sliders.append(slider)

    cols = 4
    rows_widgets = []
    for r in range(0, len(cards), cols):
        rows_widgets.append(widgets.HBox(cards[r:r+cols]))
    grid = widgets.VBox(rows_widgets)

    _grid_out.clear_output(wait=True)
    with _grid_out:
        display(widgets.HTML(f"<h3>Generation {generation} -- Rate each chair (0 = skip, 1 = hate, 5 = love)</h3>"))
        display(grid)


def _on_evolve(btn):
    global population, generation, all_rated_history

    ratings = [s.value for s in _sliders]
    for chrom, r in zip(population, ratings):
        chrom.fitness = float(r)
        chrom.fitness_details = {'user_rating': float(r)}
        all_rated_history.append((generation, copy.deepcopy(chrom)))

    rated_pop = [c for c in population if c.fitness > 0]
    rated_pop.sort(key=lambda c: c.fitness, reverse=True)

    with _status_out:
        clear_output(wait=True)
        mean = np.mean([c.fitness for c in rated_pop]) if rated_pop else 0
        best = max((c.fitness for c in rated_pop), default=0)
        print(f"Gen {generation} applied: {len(rated_pop)}/{len(population)} rated | "
              f"mean={mean:.2f}, best={best:.1f}")

    if len(rated_pop) < 2:
        with _status_out:
            print("Need at least 2 rated chairs (rating > 0) to evolve.")
        return

    pop_size = GA_CONFIG['population_size']
    elite_n  = GA_CONFIG['elitism_count']
    cx_rate  = GA_CONFIG['crossover_rate']
    mut_rate = GA_CONFIG['mutation_rate']
    tourn_k  = GA_CONFIG['tournament_size']

    next_gen = []
    for i in range(min(elite_n, len(rated_pop))):
        elite = copy.deepcopy(rated_pop[i])
        elite.fitness = 0.0; elite.fitness_details = {}
        next_gen.append(elite)

    while len(next_gen) < pop_size:
        p1 = tournament_selection(rated_pop, tourn_k)
        p2 = tournament_selection(rated_pop, tourn_k)
        if random.random() < cx_rate:
            c1, c2 = crossover_uniform(p1, p2)
        else:
            c1 = copy.deepcopy(p1); c2 = copy.deepcopy(p2)
        c1 = mutate(c1, mut_rate)
        c2 = mutate(c2, mut_rate)
        if random.random() < 0.1: c1 = mutate_swap(c1)
        if random.random() < 0.1: c2 = mutate_swap(c2)
        c1.fitness = 0.0; c1.fitness_details = {}
        c2.fitness = 0.0; c2.fitness_details = {}
        next_gen.extend([c1, c2])

    population = next_gen[:pop_size]
    generation += 1

    with _status_out:
        print(f"Generation {generation} created. Rendering new previews...")

    _refresh_grid()

    with _status_out:
        clear_output(wait=True)
        print(f"Generation {generation} ready. Rate the new chairs below.")


def _on_regenerate(btn):
    with _status_out:
        clear_output(wait=True)
        print("Regenerating previews...")
    _refresh_grid()
    with _status_out:
        clear_output(wait=True)
        print("Previews refreshed.")


def _on_reset(btn):
    global population, generation
    population = [random_chromosome() for _ in range(GA_CONFIG['population_size'])]
    generation = 0
    with _status_out:
        clear_output(wait=True)
        print("Reset to new random Generation 0.")
    _refresh_grid()


def _on_plot(btn):
    _plot_out.clear_output(wait=True)
    with _plot_out:
        if not all_rated_history:
            print("No ratings yet.")
            return
        gen_ratings = {}
        for gn, c in all_rated_history:
            gen_ratings.setdefault(gn, []).append(c.fitness)
        gens = sorted(gen_ratings.keys())
        means = [np.mean(gen_ratings[g]) for g in gens]
        maxes = [max(gen_ratings[g]) for g in gens]
        mins  = [min(gen_ratings[g]) for g in gens]

        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        ax.plot(gens, means, 'b-o', label='Mean', markersize=6)
        ax.plot(gens, maxes, 'g-^', label='Max', markersize=5)
        ax.plot(gens, mins,  'r-v', label='Min', markersize=5, alpha=0.6)
        ax.fill_between(gens, mins, maxes, alpha=0.1, color='gray')
        for g in gens:
            ax.scatter([g] * len(gen_ratings[g]), gen_ratings[g],
                       alpha=0.3, color='steelblue', s=20)
        ax.set_xlabel('Generation'); ax.set_ylabel('User Rating (0-5)')
        ax.set_title('Rating Progression'); ax.set_ylim(-0.2, 5.5)
        ax.set_xticks(gens); ax.legend(); ax.grid(True, alpha=0.3)
        fig.tight_layout(); plt.show()


def _on_winner(btn):
    _winner_out.clear_output(wait=True)
    with _winner_out:
        if not all_rated_history:
            print("No ratings yet -- nothing to show.")
            return

        best_map = {}
        for gn, c in all_rated_history:
            key = (tuple(c.genes), c.pose_anchor)
            if key not in best_map or c.fitness > best_map[key][1].fitness:
                best_map[key] = (gn, c)

        sorted_winners = sorted(best_map.values(), key=lambda x: -x[1].fitness)
        top_gn, top_chrom = sorted_winners[0]

        print(f"Winner: rating={top_chrom.fitness:.1f} (from Gen {top_gn})")
        print(f"  genes={top_chrom.genes}")
        print(f"  pose_anchor={top_chrom.pose_anchor}")

        mesh, _ = decode_chair(top_chrom, resolution=GA_CONFIG.get('final_resolution', 128))

        views = [
            ("Side",      -30,  -60),
            ("Front",     -20,  -90),
            ("Three-qtr", -35, -120),
        ]

        images = []
        for name, elev, azim in views:
            img_bytes = _mesh_to_png(mesh, figsize=(4, 4),
                                     title=f"{name}", elev=elev, azim=azim)
            images.append(widgets.Image(value=img_bytes, format='png',
                                         width=320, height=320))
        display(widgets.HTML(
            f"<h3>Final Winner -- rating {top_chrom.fitness:.1f} "
            f"(Gen {top_gn}, anchor={top_chrom.pose_anchor})</h3>"
        ))
        display(widgets.HBox(images))

        if len(sorted_winners) > 1:
            lines = ["<h4>Top-5 rated history:</h4><ul>"]
            for rank, (gn, c) in enumerate(sorted_winners[:5]):
                lines.append(
                    f"<li>#{rank+1}: rating={c.fitness:.1f} "
                    f"(Gen {gn}) genes={c.genes} anchor={c.pose_anchor}</li>"
                )
            lines.append("</ul>")
            display(widgets.HTML("".join(lines)))


btn_evolve     = widgets.Button(description='Apply Ratings & Evolve ->',
                                 button_style='success',
                                 layout=widgets.Layout(width='220px'))
btn_regenerate = widgets.Button(description='Regenerate Preview',
                                 layout=widgets.Layout(width='180px'))
btn_reset      = widgets.Button(description='New Random Population',
                                 button_style='warning',
                                 layout=widgets.Layout(width='200px'))
btn_plot       = widgets.Button(description='Show Progress Plot',
                                 button_style='info',
                                 layout=widgets.Layout(width='180px'))
btn_winner     = widgets.Button(description='Show Final Winner',
                                 button_style='danger',
                                 layout=widgets.Layout(width='180px'))

btn_evolve.on_click(_on_evolve)
btn_regenerate.on_click(_on_regenerate)
btn_reset.on_click(_on_reset)
btn_plot.on_click(_on_plot)
btn_winner.on_click(_on_winner)

controls = widgets.HBox([btn_evolve, btn_regenerate, btn_reset, btn_plot, btn_winner])

display(widgets.VBox([controls, _status_out, _grid_out, _plot_out, _winner_out]))
_refresh_grid()


## 8. Progress Plot (standalone)

The "Show Progress Plot" button in the UI does this too, but you can also run it here any time.

In [ ]:
if not all_rated_history:
    print("No ratings yet.")
else:
    # Group ratings by generation
    gen_ratings = {}
    for gen_num, chrom in all_rated_history:
        gen_ratings.setdefault(gen_num, []).append(chrom.fitness)

    gens = sorted(gen_ratings.keys())
    means = [np.mean(gen_ratings[g]) for g in gens]
    maxes = [max(gen_ratings[g]) for g in gens]
    mins  = [min(gen_ratings[g]) for g in gens]

    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    ax.plot(gens, means, 'b-o', label='Mean', markersize=7)
    ax.plot(gens, maxes, 'g-^', label='Max', markersize=6)
    ax.plot(gens, mins,  'r-v', label='Min', markersize=6, alpha=0.6)
    ax.fill_between(gens, mins, maxes, alpha=0.1, color='gray')

    # Individual points
    for g in gens:
        ax.scatter([g] * len(gen_ratings[g]), gen_ratings[g],
                   alpha=0.3, color='steelblue', s=20)

    ax.set_xlabel('Generation')
    ax.set_ylabel('User Rating (1-5)')
    ax.set_title('User Preference Progression Across Generations')
    ax.set_ylim(0, 5.5)
    ax.set_xticks(gens)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(f"\nTotal chairs rated: {len(all_rated_history)}")
    print(f"Best rating so far: {max(c.fitness for _, c in all_rated_history):.1f}")

## 9. Export Top-Rated Chairs (across all generations)

In [ ]:
TOP_K_EXPORT = 5

# Pick top-rated chairs from all history (deduplicate by genes+anchor)
seen = set()
unique_rated = []
for gen_num, chrom in sorted(all_rated_history, key=lambda x: -x[1].fitness):
    key = (tuple(chrom.genes), chrom.pose_anchor)
    if key in seen:
        continue
    seen.add(key)
    unique_rated.append((gen_num, chrom))

top = unique_rated[:TOP_K_EXPORT]

outdir = os.path.join(PARTSDF_ROOT, "notebooks", "ga_chair_generation", "output_user")
os.makedirs(outdir, exist_ok=True)

print(f"Exporting top {len(top)} chairs to {outdir}/")

for rank, (gen_num, chrom) in enumerate(top):
    print(f"\nRank #{rank+1} | gen={gen_num} | rating={chrom.fitness:.1f}")
    print(f"  genes={chrom.genes}, anchor={chrom.pose_anchor}")

    # High-res mesh
    lat, R, t, s = chromosome_to_latent_pose(chrom)
    mesh = create_mesh(
        model, lat, GA_CONFIG['final_resolution'], 32**3,
        grid_filler=grid_filler, device=device, R=R, t=t, s=s,
    )
    parts = create_parts(
        model, lat, GA_CONFIG['final_resolution'], 32**3,
        grid_filler=grid_filler, device=device, R=R, t=t, s=s,
    )

    if not mesh.is_empty:
        mesh.export(os.path.join(outdir, f"user_rank{rank+1}_full.obj"))
    colored = trimesh.util.concatenate(get_color_parts(parts))
    if not colored.is_empty:
        colored.export(os.path.join(outdir, f"user_rank{rank+1}_parts.obj"))

    info = {
        'rank': rank + 1,
        'generation': gen_num,
        'user_rating': chrom.fitness,
        'genes': chrom.genes,
        'pose_anchor': chrom.pose_anchor,
        'mode': MIXING_MODE,
    }
    with open(os.path.join(outdir, f"user_rank{rank+1}_info.json"), 'w') as f:
        json.dump(info, f, indent=2)

print(f"\nDone! Outputs in {outdir}/")

## 10. Save / Load Session (optional)

Save the current population & rating history so you can resume later.

In [ ]:
SESSION_FILE = os.path.join(
    PARTSDF_ROOT, "notebooks", "ga_chair_generation", "user_session.json"
)

def save_session():
    data = {
        'generation': generation,
        'population': [
            {'genes': c.genes, 'pose_anchor': c.pose_anchor,
             'fitness': c.fitness, 'fitness_details': c.fitness_details}
            for c in population
        ],
        'history': [
            {'gen': g, 'genes': c.genes, 'pose_anchor': c.pose_anchor, 'fitness': c.fitness}
            for g, c in all_rated_history
        ],
        'config': GA_CONFIG,
        'mode': MIXING_MODE,
    }
    with open(SESSION_FILE, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Saved to {SESSION_FILE}")

def load_session():
    global generation, population, all_rated_history
    with open(SESSION_FILE) as f:
        data = json.load(f)
    generation = data['generation']
    population = [
        Chromosome(genes=d['genes'], pose_anchor=d['pose_anchor'],
                   fitness=d['fitness'], fitness_details=d.get('fitness_details', {}))
        for d in data['population']
    ]
    all_rated_history = [
        (d['gen'], Chromosome(genes=d['genes'], pose_anchor=d['pose_anchor'],
                               fitness=d['fitness']))
        for d in data['history']
    ]
    print(f"Loaded: generation={generation}, pop={len(population)}, "
          f"history={len(all_rated_history)} entries")

# Uncomment to use:
# save_session()
# load_session()

## How to use

1. **Run cells 0–6** once to set everything up.
2. **Run cell 7** → an interactive panel appears with 8 chair previews + sliders.
3. **Move each slider** to rate chairs 0–5 (0 = skip, 5 = love).
4. Click **"Apply Ratings & Evolve →"** → next generation auto-renders.
5. Repeat step 3–4 as many times as you like.
6. Click **"Show Progress Plot"** (or run cell 8) anytime to see rating trends.
7. When satisfied, run **cell 9** to export top chairs as OBJ files.
8. (Optional) **cell 10** to save/load session state.

### Button reference
- **Apply Ratings & Evolve →** : take current slider values → make next generation
- **Regenerate Preview** : re-render chairs (if you change `preview_resolution` etc.)
- **New Random Population** : reset to fresh Gen 0 (keeps rating history)
- **Show Progress Plot** : rating trend across generations

### Tips
- **5** on chairs you want to see more of (their parts get reused).
- **1–2** on ugly/broken chairs.
- **0** to skip (won't count toward selection).
- Expect 5–10 generations before designs look refined.
- Tune `mutation_rate` in `GA_CONFIG` for diversity control.